# Stage 04: Data Acquisition & Ingestion
One API pull (yfinance, AAPL daily prices) and one scraped table (Wikipedia, S&P 500 constituents), each validated and saved to `data/raw/` with a reproducible, timestamped filename.

## Data Sources & Documentation

**Source 1 (API): Yahoo Finance via `yfinance`**
- Ticker: `AAPL`
- Endpoint: yfinance's `download()` wraps Yahoo Finance's public historical-data endpoint.
- Params: `period='6mo'`, `interval='1d'`
- API key: not required for this source. The `.env`/`config.py` pattern is still demonstrated below (loading `API_KEY` if present) in case a future data source needs one, but yfinance's public endpoint works without authentication.

**Source 2 (Scrape): Wikipedia — List of S&P 500 companies**
- URL: `https://en.wikipedia.org/wiki/List_of_S%26P_500_companies`
- Table scraped: the first table on the page (`Symbol`, `Security`, `GICS Sector`, `GICS Sub-Industry`, etc.)
- Public page, permitted for reasonable, low-frequency educational scraping. No login or paywall.

**Validation logic (both sources):** required columns present, shape reported, NA counts per column — implemented once in `src/validate.py` and reused for both datasets.

## Setup

In [ ]:
import sys
sys.path.append('../src')

import os
from datetime import datetime

import pandas as pd
import requests
import yfinance as yf
from bs4 import BeautifulSoup

from config import load_env, get_key
from validate import validate_dataframe

load_env()

# confirm whether an API key is set, without printing its actual value
api_key = get_key('API_KEY', required=False)
print('API_KEY present:', api_key is not None)
print('(Not required for this notebook\'s sources, but demonstrating the pattern.)')

## 1. API Pull: AAPL daily prices via yfinance

In [ ]:
TICKER = 'AAPL'

try:
    api_df = yf.download(TICKER, period='6mo', interval='1d')
    api_df = api_df.reset_index()
    # yfinance sometimes returns multi-level columns when downloading a single ticker
    # this flattens them if needed
    if isinstance(api_df.columns, pd.MultiIndex):
        api_df.columns = [c[0] if c[1] == '' else c[0] for c in api_df.columns]
    print(f'Pulled {len(api_df)} rows for {TICKER}')
except Exception as e:
    print(f'API pull failed: {e}')
    api_df = pd.DataFrame()

api_df.head()

In [ ]:
# parse dtypes explicitly
api_df['Date'] = pd.to_datetime(api_df['Date'])
for col in ['Open', 'High', 'Low', 'Close', 'Volume']:
    if col in api_df.columns:
        api_df[col] = pd.to_numeric(api_df[col], errors='coerce')

api_df.dtypes

In [ ]:
api_required_cols = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
api_report = validate_dataframe(api_df, required_columns=api_required_cols, name=f'API ({TICKER})')

In [ ]:
timestamp = datetime.now().strftime('%Y%m%d-%H%M')
api_filename = f'../data/raw/api_yfinance_{TICKER}_{timestamp}.csv'
api_df.to_csv(api_filename, index=False)
print(f'Saved to {api_filename}')

## 2. Scrape: S&P 500 constituents table (Wikipedia)

In [ ]:
SCRAPE_URL = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'

try:
    response = requests.get(SCRAPE_URL, headers={'User-Agent': 'Mozilla/5.0 (educational project)'})
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'lxml')

    table = soup.find('table', {'id': 'constituents'})
    if table is None:
        # fallback in case the page structure changes and the id is different
        table = soup.find('table', {'class': 'wikitable'})

    headers = [th.get_text(strip=True) for th in table.find_all('th')]

    rows = []
    for tr in table.find_all('tr')[1:]:
        cells = [td.get_text(strip=True) for td in tr.find_all('td')]
        if cells:
            rows.append(cells)

    scrape_df = pd.DataFrame(rows, columns=headers[:len(rows[0])])
    print(f'Scraped {len(scrape_df)} rows')
except Exception as e:
    print(f'Scrape failed: {e}')
    scrape_df = pd.DataFrame()

scrape_df.head()

In [ ]:
# quick check on column names actually pulled, since Wikipedia table headers can shift over time
print(scrape_df.columns.tolist())

In [ ]:
# validate. Adjust required_columns below if the printed column list above
# differs from what's expected (Wikipedia occasionally renames headers).
scrape_required_cols = ['Symbol', 'Security', 'GICS Sector']
scrape_report = validate_dataframe(scrape_df, required_columns=scrape_required_cols, name='Scrape (S&P 500 list)')

In [ ]:
scrape_filename = f'../data/raw/scrape_wikipedia_sp500-constituents_{timestamp}.csv'
scrape_df.to_csv(scrape_filename, index=False)
print(f'Saved to {scrape_filename}')

## Assumptions & Risks

- **yfinance reliability:** yfinance wraps an unofficial Yahoo Finance endpoint, not a documented public API, so it can break or rate-limit without notice. The `try/except` above degrades gracefully to an empty DataFrame rather than crashing the notebook, but a production pipeline would need a proper retry/alerting strategy.
- **Wikipedia table structure:** the S&P 500 table's column names and structure have changed before and could change again. The scrape includes a fallback selector and a printed column list specifically so a broken selector is caught immediately rather than silently producing garbage data.
- **No API key needed for either source:** this keeps the assignment simple, but it also means neither source has guaranteed uptime/SLA the way a paid, authenticated API would.
- **`.env` is not committed:** confirmed via `.gitignore`; only `.env.example` (with a dummy placeholder key) is committed, so no real secrets are exposed even though this particular notebook doesn't require one.
- **Freshness:** both files are timestamped at pull time (`YYYYMMDD-HHMM`), so re-running this notebook later produces new files rather than overwriting old ones — useful for tracking how source data changes over time, but means `data/raw/` will accumulate files if run repeatedly.